# Study 01 — Executive Report

## Has Morocco's Minimum Wage Protected Workers Against Inflation Since 2000?

### Morocco Economic Intelligence Lab (MEIL)

---

## Objective

This notebook transforms the analytical results into an executive-ready report.

The output is a structured summary that can be shared with decision-makers, uploaded to GitHub, or later converted into PDF.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
SUMMARY_DIR = PROJECT_ROOT / "executive-summary"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

FEATURE_FILE = PROCESSED_DATA / "smig_cpi_features.csv"
RESULTS_FILE = PROCESSED_DATA / "study_results.csv"
REPORT_FILE = SUMMARY_DIR / "study_01_executive_summary.md"

SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

print("Feature file :", FEATURE_FILE)
print("Results file  :", RESULTS_FILE)
print("Report file   :", REPORT_FILE)

Feature file : c:\Users\oatta\Desktop\Economic Research Projects\Morocco-Economic-Intelligence-Lab\living-standards\study-01-smig-vs-inflation\data\processed\smig_cpi_features.csv
Results file  : c:\Users\oatta\Desktop\Economic Research Projects\Morocco-Economic-Intelligence-Lab\living-standards\study-01-smig-vs-inflation\data\processed\study_results.csv
Report file   : c:\Users\oatta\Desktop\Economic Research Projects\Morocco-Economic-Intelligence-Lab\living-standards\study-01-smig-vs-inflation\executive-summary\study_01_executive_summary.md


In [3]:
df = pd.read_csv(FEATURE_FILE)
results_df = pd.read_csv(RESULTS_FILE)

df.columns = [col.strip().lower() for col in df.columns]
results = results_df.iloc[0].to_dict()

df.head()

,year,cpi,smig_monthly,smig_index,cpi_index,inflation_rate,nominal_wage_growth,real_wage_index,real_wage_growth,purchasing_power_gap,pppr,cumulative_inflation,cumulative_wage_growth
0,2000,83.649753,1826,100.000000,100.000000,NaN,NaN,100.000000,NaN,NaN,1.000000,0.000000,0.000000
1,2001,84.168216,1826,100.000000,100.619802,0.619802,0.000000,99.384016,-0.619802,-0.619802,0.993840,0.619802,0.000000
2,2002,86.521239,1826,100.000000,103.432749,2.795620,0.000000,96.681178,-2.795620,-2.795620,0.966812,3.432749,0.000000
3,2003,87.531577,1826,100.000000,104.640568,1.167734,0.000000,95.565231,-1.167734,-1.167734,0.955652,4.640568,0.000000
4,2004,88.838812,1845,101.040526,106.203316,1.493444,1.040526,95.138767,-0.452918,-0.452918,0.951388,6.203316,1.040526


## Data Validation

In [4]:
required_columns = {
    "year",
    "cpi",
    "smig_monthly",
    "smig_index",
    "cpi_index",
    "inflation_rate",
    "nominal_wage_growth",
    "real_wage_index",
    "real_wage_growth",
    "purchasing_power_gap",
    "pppr",
    "cumulative_inflation",
    "cumulative_wage_growth"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns in feature dataset: {missing_columns}")

print("All required columns are available.")

All required columns are available.


## Study Information

In [5]:
start_year = int(df["year"].min())
end_year = int(df["year"].max())
n_obs = len(df)

study_period = f"{start_year}-{end_year}"

print("Study period :", study_period)
print("Observations  :", n_obs)

Study period : 2000-2024
Observations  : 25


## Executive Summary

In [6]:
average_inflation = df["inflation_rate"].mean()
average_nominal_growth = df["nominal_wage_growth"].mean()
average_real_growth = df["real_wage_growth"].mean()
average_pppr = df["pppr"].mean()

years_above = int((df["purchasing_power_gap"] > 0).sum())
years_below = int((df["purchasing_power_gap"] < 0).sum())
years_equal = int((df["purchasing_power_gap"] == 0).sum())

best_year_row = df.loc[df["purchasing_power_gap"].idxmax()]
worst_year_row = df.loc[df["purchasing_power_gap"].idxmin()]

cumulative_nominal = df["cumulative_wage_growth"].iloc[-1]
cumulative_inflation = df["cumulative_inflation"].iloc[-1]
cumulative_real = cumulative_nominal - cumulative_inflation

if cumulative_real > 0:
    cumulative_message = (
        "Over the full period, the statutory minimum wage increased faster "
        "than consumer prices in cumulative terms."
    )
else:
    cumulative_message = (
        "Over the full period, consumer prices increased faster than the "
        "statutory minimum wage in cumulative terms."
    )

executive_summary = f"""
This study examines whether Morocco's statutory minimum wage protected workers
against inflation over the period {study_period}.

The analysis shows an average annual inflation rate of {average_inflation:.2f}%, 
compared with an average nominal minimum wage growth rate of 
{average_nominal_growth:.2f}%.

The average real wage growth rate was {average_real_growth:.2f}%, while the
average Purchasing Power Protection Ratio (PPPR) stood at {average_pppr:.3f}.

The minimum wage outpaced inflation in {years_above} years, while inflation
outpaced the minimum wage in {years_below} years. In {years_equal} years, both
grew at the same pace.

The best year for purchasing power was {int(best_year_row['year'])}, with a gap
of {best_year_row['purchasing_power_gap']:.2f} percentage points.

The worst year for purchasing power was {int(worst_year_row['year'])}, with a
gap of {worst_year_row['purchasing_power_gap']:.2f} percentage points.

{cumulative_message}
"""

print(executive_summary)


This study examines whether Morocco's statutory minimum wage protected workers
against inflation over the period 2000-2024.

The analysis shows an average annual inflation rate of 1.85%, 
compared with an average nominal minimum wage growth rate of 
2.28%.

The average real wage growth rate was 0.43%, while the
average Purchasing Power Protection Ratio (PPPR) stood at 1.026.

The minimum wage outpaced inflation in 8 years, while inflation
outpaced the minimum wage in 16 years. In 0 years, both
grew at the same pace.

The best year for purchasing power was 2011, with a gap
of 9.03 percentage points.

The worst year for purchasing power was 2006, with a
gap of -3.28 percentage points.

Over the full period, the statutory minimum wage increased faster than consumer prices in cumulative terms.



## Key Findings

In [7]:
key_findings = []

if years_above > years_below:
    key_findings.append(
        "The minimum wage outpaced inflation in more years than it lagged behind inflation."
    )
else:
    key_findings.append(
        "Inflation outpaced the minimum wage in more years than the minimum wage outpaced inflation."
    )

if cumulative_real > 0:
    key_findings.append(
        "The cumulative real wage performance remained positive over the full 2000–2024 period."
    )
else:
    key_findings.append(
        "The cumulative real wage performance turned negative over the full 2000–2024 period."
    )

if average_pppr > 1:
    key_findings.append(
        "On average, the minimum wage protected purchasing power better than inflation eroded it."
    )
else:
    key_findings.append(
        "On average, inflation eroded purchasing power faster than the minimum wage protected it."
    )

key_findings.append(
    f"The strongest purchasing-power gain occurred in {int(best_year_row['year'])}."
)

key_findings.append(
    f"The weakest purchasing-power outcome occurred in {int(worst_year_row['year'])}."
)

for i, item in enumerate(key_findings, start=1):
    print(f"{i}. {item}")

1. Inflation outpaced the minimum wage in more years than the minimum wage outpaced inflation.
2. The cumulative real wage performance remained positive over the full 2000–2024 period.
3. On average, the minimum wage protected purchasing power better than inflation eroded it.
4. The strongest purchasing-power gain occurred in 2011.
5. The weakest purchasing-power outcome occurred in 2006.


## Statistical Highlights

In [8]:
stats_summary = pd.DataFrame([
    {
        "Indicator": "Study period",
        "Value": study_period
    },
    {
        "Indicator": "Observations",
        "Value": n_obs
    },
    {
        "Indicator": "Average inflation rate (%)",
        "Value": round(average_inflation, 2)
    },
    {
        "Indicator": "Average nominal wage growth (%)",
        "Value": round(average_nominal_growth, 2)
    },
    {
        "Indicator": "Average real wage growth (%)",
        "Value": round(average_real_growth, 2)
    },
    {
        "Indicator": "Average PPPR",
        "Value": round(average_pppr, 3)
    },
    {
        "Indicator": "Years above inflation",
        "Value": years_above
    },
    {
        "Indicator": "Years below inflation",
        "Value": years_below
    },
    {
        "Indicator": "Years equal to inflation",
        "Value": years_equal
    },
    {
        "Indicator": "Best year",
        "Value": int(best_year_row["year"])
    },
    {
        "Indicator": "Worst year",
        "Value": int(worst_year_row["year"])
    },
    {
        "Indicator": "Cumulative nominal wage growth (%)",
        "Value": round(cumulative_nominal, 2)
    },
    {
        "Indicator": "Cumulative inflation (%)",
        "Value": round(cumulative_inflation, 2)
    },
    {
        "Indicator": "Cumulative real gain (%)",
        "Value": round(cumulative_real, 2)
    }
])

display(stats_summary)

,Indicator,Value
0,Study period,2000-2024
1,Observations,25
2,Average inflation rate (%),1.85
3,Average nominal wage growth (%),2.28
4,Average real wage growth (%),0.43
5,Average PPPR,1.026
6,Years above inflation,8
7,Years below inflation,16
8,Years equal to inflation,0
9,Best year,2011


## Decade Comparison

In [9]:
df["decade"] = (df["year"] // 10) * 10

decade_summary = (
    df.groupby("decade")[[
        "inflation_rate",
        "nominal_wage_growth",
        "real_wage_growth",
        "pppr"
    ]]
    .mean()
    .round(2)
    .reset_index()
)

decade_summary.columns = [
    "Decade",
    "Average Inflation (%)",
    "Average Nominal Wage Growth (%)",
    "Average Real Wage Growth (%)",
    "Average PPPR"
]

display(decade_summary)

,Decade,Average Inflation (%),Average Nominal Wage Growth (%),Average Real Wage Growth (%),Average PPPR
0,2000,1.90,1.21,-0.68,0.95
1,2010,1.16,2.93,1.77,1.06
2,2020,3.17,2.91,-0.26,1.13


## Best and Worst Years

In [10]:
extremes = pd.DataFrame([
    {
        "Type": "Best year",
        "Year": int(best_year_row["year"]),
        "Purchasing Power Gap": round(best_year_row["purchasing_power_gap"], 2),
        "Real Wage Growth": round(best_year_row["real_wage_growth"], 2)
    },
    {
        "Type": "Worst year",
        "Year": int(worst_year_row["year"]),
        "Purchasing Power Gap": round(worst_year_row["purchasing_power_gap"], 2),
        "Real Wage Growth": round(worst_year_row["real_wage_growth"], 2)
    }
])

display(extremes)

,Type,Year,Purchasing Power Gap,Real Wage Growth
0,Best year,2011,9.03,9.03
1,Worst year,2006,-3.28,-3.28


## Figures Included in This Study

In [11]:
figure_files = sorted(FIGURES_DIR.glob("*.png"))

for file in figure_files:
    print(file.name)

figure_01_executive_dashboard.png
figure_01_smig_vs_cpi.png
figure_02_base100.png
figure_02_base100_comparison.png
figure_03_inflation.png
figure_03_purchasing_power_gap.png
figure_04_pppr.png
figure_04_wage_growth.png
figure_05_inflation_vs_wage_growth.png
figure_05_real_wage.png
figure_06_cumulative_growth.png
figure_06_gap.png
figure_07_pppr.png
figure_07_revaluation_timeline.png
figure_08_economic_heatmap.png


## Assemble Executive Report

In [13]:
figure_links = {
    "figure_01_executive_dashboard.png": "../outputs/figures/figure_01_executive_dashboard.png",
    "figure_02_base100_comparison.png": "../outputs/figures/figure_02_base100_comparison.png",
    "figure_03_purchasing_power_gap.png": "../outputs/figures/figure_03_purchasing_power_gap.png",
    "figure_04_pppr.png": "../outputs/figures/figure_04_pppr.png",
    "figure_05_inflation_vs_wage_growth.png": "../outputs/figures/figure_05_inflation_vs_wage_growth.png",
    "figure_06_cumulative_growth.png": "../outputs/figures/figure_06_cumulative_growth.png",
    "figure_07_revaluation_timeline.png": "../outputs/figures/figure_07_revaluation_timeline.png",
    "figure_08_economic_heatmap.png": "../outputs/figures/figure_08_economic_heatmap.png",
}

report_lines = []

report_lines.append("# Study 01 — Executive Summary")
report_lines.append("")
report_lines.append("## Has Morocco's Minimum Wage Protected Workers Against Inflation Since 2000?")
report_lines.append("")
report_lines.append("### Morocco Economic Intelligence Lab (MEIL)")
report_lines.append("")
report_lines.append("## Executive Summary")
report_lines.append("")
report_lines.append(executive_summary.strip())
report_lines.append("")
report_lines.append("## Key Findings")
report_lines.append("")
for item in key_findings:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Statistical Highlights")
report_lines.append("")
report_lines.append(stats_summary.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Decade Comparison")
report_lines.append("")
report_lines.append(decade_summary.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Best and Worst Years")
report_lines.append("")
report_lines.append(extremes.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Figures")
report_lines.append("")
for name, rel_path in figure_links.items():
    report_lines.append(f"### {name}")
    report_lines.append(f"![{name}]({rel_path})")
    report_lines.append("")
report_lines.append("## Conclusion")
report_lines.append("")
report_lines.append(
    "The analysis indicates that Morocco's statutory minimum wage generally "
    "kept pace with inflation over the full study period, but the protection "
    "of purchasing power was uneven across years."
)
report_lines.append("")
report_lines.append("## Limitations")
report_lines.append("")
report_lines.append(
    "- The minimum wage is only a proxy for workers at the lower end of the formal labour market."
)
report_lines.append(
    "- The analysis does not capture informal employment dynamics."
)
report_lines.append(
    "- CPI measures average price changes and may differ from the consumption basket of individual households."
)
report_lines.append(
    "- The study is annual and does not capture intra-year timing effects of wage adjustments and price shocks."
)
report_lines.append("")
report_lines.append("## Future Work")
report_lines.append("")
report_lines.append(
    "- Compare the minimum wage with other wage benchmarks."
)
report_lines.append(
    "- Extend the analysis to sectoral wages and household income measures."
)
report_lines.append(
    "- Add a regional dimension when data become available."
)

report_md = "\n".join(report_lines)

print(report_md[:2500])

# Study 01 — Executive Summary

## Has Morocco's Minimum Wage Protected Workers Against Inflation Since 2000?

### Morocco Economic Intelligence Lab (MEIL)

## Executive Summary

This study examines whether Morocco's statutory minimum wage protected workers
against inflation over the period 2000-2024.

The analysis shows an average annual inflation rate of 1.85%, 
compared with an average nominal minimum wage growth rate of 
2.28%.

The average real wage growth rate was 0.43%, while the
average Purchasing Power Protection Ratio (PPPR) stood at 1.026.

The minimum wage outpaced inflation in 8 years, while inflation
outpaced the minimum wage in 16 years. In 0 years, both
grew at the same pace.

The best year for purchasing power was 2011, with a gap
of 9.03 percentage points.

The worst year for purchasing power was 2006, with a
gap of -3.28 percentage points.

Over the full period, the statutory minimum wage increased faster than consumer prices in cumulative terms.

## Key Findings

- 

In [14]:
with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Executive report exported successfully to: {REPORT_FILE}")

Executive report exported successfully to: c:\Users\oatta\Desktop\Economic Research Projects\Morocco-Economic-Intelligence-Lab\living-standards\study-01-smig-vs-inflation\executive-summary\study_01_executive_summary.md


## Report Preview

In [15]:
print(report_md[:4000])

# Study 01 — Executive Summary

## Has Morocco's Minimum Wage Protected Workers Against Inflation Since 2000?

### Morocco Economic Intelligence Lab (MEIL)

## Executive Summary

This study examines whether Morocco's statutory minimum wage protected workers
against inflation over the period 2000-2024.

The analysis shows an average annual inflation rate of 1.85%, 
compared with an average nominal minimum wage growth rate of 
2.28%.

The average real wage growth rate was 0.43%, while the
average Purchasing Power Protection Ratio (PPPR) stood at 1.026.

The minimum wage outpaced inflation in 8 years, while inflation
outpaced the minimum wage in 16 years. In 0 years, both
grew at the same pace.

The best year for purchasing power was 2011, with a gap
of 9.03 percentage points.

The worst year for purchasing power was 2006, with a
gap of -3.28 percentage points.

Over the full period, the statutory minimum wage increased faster than consumer prices in cumulative terms.

## Key Findings

- 

In [16]:
print("=" * 70)
print("EXECUTIVE REPORT COMPLETED")
print("=" * 70)
print()
print(f"Study period : {study_period}")
print(f"Report file   : {REPORT_FILE}")
print(f"Figures found : {len(figure_files)}")
print()
print("Notebook 07 completed successfully.")

EXECUTIVE REPORT COMPLETED

Study period : 2000-2024
Report file   : c:\Users\oatta\Desktop\Economic Research Projects\Morocco-Economic-Intelligence-Lab\living-standards\study-01-smig-vs-inflation\executive-summary\study_01_executive_summary.md
Figures found : 15

Notebook 07 completed successfully.
